In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into DataFrames (store for later steps)
metric_df = pd.read_csv('metric.csv', low_memory=False)
trace_df = pd.read_csv('trace.csv', low_memory=False)
log_df = pd.read_csv('log.csv', low_memory=False)
error_df = pd.read_csv('error_logs.csv', low_memory=False)

# Helper to convert unix-second min/max to tz-aware datetimes (UTC->Asia/Shanghai)
def to_tz_str(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz)

# ---- metric.csv summary ----
metric_total_rows = len(metric_df)
metric_min_ts = metric_df['timestamp'].min() if 'timestamp' in metric_df.columns else None
metric_max_ts = metric_df['timestamp'].max() if 'timestamp' in metric_df.columns else None
metric_unique_cmdb = metric_df['cmdb_id'].nunique() if 'cmdb_id' in metric_df.columns else 0

metric_summary = pd.DataFrame([{
    'total_rows': metric_total_rows,
    'min_timestamp_utc8': to_tz_str(metric_min_ts),
    'max_timestamp_utc8': to_tz_str(metric_max_ts),
    'unique_cmdb_id_count': metric_unique_cmdb
}])

metric_top_cmdb = metric_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
metric_top_pairs = (
    metric_df.groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index(name='count')
)

# ---- trace.csv summary ----
trace_total_rows = len(trace_df)
trace_min_ts = trace_df['timestamp'].min() if 'timestamp' in trace_df.columns else None
trace_max_ts = trace_df['timestamp'].max() if 'timestamp' in trace_df.columns else None
trace_unique_cmdb = trace_df['cmdb_id'].nunique() if 'cmdb_id' in trace_df.columns else 0

trace_summary = pd.DataFrame([{
    'total_rows': trace_total_rows,
    'min_timestamp_utc8': to_tz_str(trace_min_ts),
    'max_timestamp_utc8': to_tz_str(trace_max_ts),
    'unique_cmdb_id_count': trace_unique_cmdb
}])

trace_top_cmdb = trace_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
trace_top_pairs = (
    trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)
    .size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index(name='count')
)

# ---- log.csv summary ----
log_total_rows = len(log_df)
log_min_ts = log_df['timestamp'].min() if 'timestamp' in log_df.columns else None
log_max_ts = log_df['timestamp'].max() if 'timestamp' in log_df.columns else None
log_unique_cmdb = log_df['cmdb_id'].nunique() if 'cmdb_id' in log_df.columns else 0

log_summary = pd.DataFrame([{
    'total_rows': log_total_rows,
    'min_timestamp_utc8': to_tz_str(log_min_ts),
    'max_timestamp_utc8': to_tz_str(log_max_ts),
    'unique_cmdb_id_count': log_unique_cmdb
}])

log_top_cmdb = log_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')
log_top_pairs = (
    log_df.groupby(['cmdb_id', 'log_name'], dropna=False)
    .size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index(name='count')
)

# ---- error_logs.csv summary and preview ----
error_total_rows = len(error_df)
error_min_ts = error_df['timestamp'].min() if 'timestamp' in error_df.columns else None
error_max_ts = error_df['timestamp'].max() if 'timestamp' in error_df.columns else None
error_unique_cmdb = error_df['cmdb_id'].nunique() if 'cmdb_id' in error_df.columns else 0

error_summary = pd.DataFrame([{
    'total_rows': error_total_rows,
    'min_timestamp_utc8': to_tz_str(error_min_ts),
    'max_timestamp_utc8': to_tz_str(error_max_ts),
    'unique_cmdb_id_count': error_unique_cmdb
}])

error_top_cmdb = error_df['cmdb_id'].value_counts().head(20).rename_axis('cmdb_id').reset_index(name='count')

# Prepare a compact preview (up to 20 rows). Convert timestamp to utc+8 datetime for readability.
if 'timestamp' in error_df.columns:
    error_preview = error_df.copy()
    error_preview['timestamp_utc8'] = pd.to_datetime(error_preview['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    error_preview = error_preview[['timestamp_utc8', 'cmdb_id', 'message']].head(20)
else:
    error_preview = error_df[['cmdb_id', 'message']].head(20)

# Final compact outputs (variables displayed for IPython)
metric_summary, metric_top_cmdb, metric_top_pairs, trace_summary, trace_top_cmdb, trace_top_pairs, log_summary, log_top_cmdb, log_top_pairs, error_summary, error_top_cmdb, error_preview

```
Out[1]:
```
summary = (
    "Telemetry summary (straightforward):\n"
    "- metric.csv: 145,812 rows covering 2022-03-21 00:30:00+08 to 2022-03-21 00:59:00+08, 59 unique cmdb_id. "
    "Top cmdb_ids by row count include adservice (10,230), adservice2 (9,990), and multiple frontend and service pods.\n"
    "- trace.csv: 29,392 rows covering the same minute range, 40 unique cmdb_id. "
    "Top trace contributors include frontend pods and recommendationservice2-0 with many trace.* series.\n"
    "- log.csv: 1,700 rows over the same minute range, 31 unique cmdb_id. "
    "Log rows are distributed across many service pods; common log_names are log.error_count and log.row_count.\n"
    "- error_logs.csv: 527 rows from 2022-03-21 00:30:01+08 to 2022-03-21 00:59:58+08, 3 unique cmdb_id. "
    "Errors are concentrated on frontend pods (frontend-0: 222, frontend-1: 158, frontend-2: 147). The preview shows repeated warnings like "
    "'severity: warning, message: failed to retrieve...'.\n\n"
    "Key observation: although metric and trace data span many services (notably adservice/adservice2 and various back-end services), "
    "the error log activity is highly concentrated on frontend pods with repeated 'failed to retrieve' warnings during the 00:30–00:59 window. "
    "This suggests the immediate visible client-facing issue is with the frontend pods; further investigation should focus on frontend logs, "
    "frontend-related metrics (CPU/memory/network), and upstream calls from frontends to back-end services during that time window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows        min_timestamp_utc8        max_timestamp_utc8  unique_cmdb_id_count
0      145812 2022-03-21 00:30:00+08:00 2022-03-21 00:59:00+08:00                    59,                      cmdb_id  count
0                  adservice  10230
1                 adservice2   9990
2                frontend2-0   5040
3                 frontend-0   4800
4                 frontend-1   4680
5                 frontend-2   4500
6          checkoutservice-0   3780
7          checkoutservice-1   3540
8          checkoutservice-2   3540
9    productcatalogservice-0   3480
10   productcatalogservice-2   3480
11   productcatalogservice-1   3360
12        checkoutservice2-0   3300
13              adservice2-0   3060
14  productcatalogservice2-0   2880
15             cartservice-1   2880
16             cartservice-0   2760
17             cartservice-2   2640
18            cartservice2-0   2640
19               adservice-0   2580,                cmdb_id                                           kpi_name  count
0   shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     30
1            adservice                                     app.grpc.count     30
2            adservice                                       app.grpc.mrt     30
3            adservice                                        app.grpc.rr     30
4            adservice                                        app.grpc.sr     30
5            adservice                                     app.http.count     30
6   shippingservice2-0  container.node-5.container_network_receive_MB....     30
7   shippingservice2-0  container.node-5.container_network_receive_err...     30
8   shippingservice2-0  container.node-5.container_network_receive_pac...     30
9   shippingservice2-0  container.node-5.container_network_receive_pac...     30
10  shippingservice2-0  container.node-5.container_network_transmit_MB...     30
11  shippingservice2-0  container.node-5.container_network_transmit_er...     30
12  shippingservice2-0  container.node-5.container_network_transmit_pa...     30
13  shippingservice2-0  container.node-5.container_network_transmit_pa...     30
14  shippingservice2-0  container.node-5.container_memory_failures.hie...     30
15  shippingservice2-0  container.node-5.container_memory_failures.hie...     30
16  shippingservice2-0      container.node-5.container_memory_mapped_file     30
17  shippingservice2-0     container.node-5.container_memory_max_usage_MB     30
18  shippingservice2-0              container.node-5.container_memory_rss     30
19  shippingservice2-0             container.node-5.container_memory_swap     30,    total_rows        min_timestamp_utc8        max_timestamp_utc8  unique_cmdb_id_count
0       29392 2022-03-21 00:30:00+08:00 2022-03-21 00:59:00+08:00                    40,                     cmdb_id  count
0                frontend-0   2556
1                frontend-1   2456
2                frontend-2   2456
3         checkoutservice-2   1604
4         checkoutservice-1   1552
5         checkoutservice-0   1528
6   productcatalogservice-0   1072
7               frontend2-0   1064
8   productcatalogservice-1   1060
9   productcatalogservice-2   1048
10       checkoutservice2-0    972
11  recommendationservice-0    840
12  recommendationservice-2    840
13  recommendationservice-1    840
14            cartservice-0    716
15            cartservice-1    704
16            cartservice-2    704
17        currencyservice-1    640
18        currencyservice-0    636
19        currencyservice-2    628,                      cmdb_id                                      trace_name  count
0   recommendationservice2-0  trace.to_productcatalogservice-1.duration_mean     30
1   recommendationservice2-0   trace.to_productcatalogservice-1.duration_p95     30
2   recommendationservice2-0     trace.to_productcatalogservice-1.error_rate     30
3   recommendationservice2-0      trace.to_productcatalogservice-1.row_count     30
4   recommendationservice2-0  trace.to_productcatalogservice-2.duration_mean     30
5   recommendationservice2-0   trace.to_productcatalogservice-2.duration_p95     30
6   recommendationservice2-0     trace.to_productcatalogservice-2.error_rate     30
7   recommendationservice2-0      trace.to_productcatalogservice-2.row_count     30
8   recommendationservice2-0                        trace.self.duration_mean     30
9   recommendationservice2-0                         trace.self.duration_p95     30
10  recommendationservice2-0                           trace.self.error_rate     30
11  recommendationservice2-0                            trace.self.row_count     30
12  recommendationservice2-0  trace.to_productcatalogservice-0.duration_mean     30
13  recommendationservice2-0   trace.to_productcatalogservice-0.duration_p95     30
14  recommendationservice2-0     trace.to_productcatalogservice-0.error_rate     30
15  recommendationservice2-0      trace.to_productcatalogservice-0.row_count     30
16   recommendationservice-2  trace.to_productcatalogservice-2.duration_mean     30
17   recommendationservice-2   trace.to_productcatalogservice-2.duration_p95     30
18   recommendationservice-2     trace.to_productcatalogservice-2.error_rate     30
19   recommendationservice-2      trace.to_productcatalogservice-2.row_count     30,    total_rows        min_timestamp_utc8        max_timestamp_utc8  unique_cmdb_id_count
0        1700 2022-03-21 00:30:00+08:00 2022-03-21 00:59:00+08:00                    31,                     cmdb_id  count
0               adservice-0     60
1               adservice-1     60
2               adservice-2     60
3             cartservice-0     60
4             cartservice-1     60
5             cartservice-2     60
6         currencyservice-1     60
7                frontend-0     60
8         currencyservice-0     60
9         currencyservice-2     60
10  productcatalogservice-2     60
11  productcatalogservice-0     60
12  productcatalogservice-1     60
13               frontend-2     60
14               frontend-1     60
15        shippingservice-2     60
16        shippingservice-1     60
17        shippingservice-0     60
18  recommendationservice-1     60
19  recommendationservice-2     60,                     cmdb_id         log_name  count
0               adservice-0  log.error_count     30
1               adservice-0    log.row_count     30
2               adservice-1  log.error_count     30
3               adservice-1    log.row_count     30
4               adservice-2  log.error_count     30
5               adservice-2    log.row_count     30
6             cartservice-0  log.error_count     30
7             cartservice-0    log.row_count     30
8             cartservice-1  log.error_count     30
9             cartservice-1    log.row_count     30
10            cartservice-2  log.error_count     30
11            cartservice-2    log.row_count     30
12        currencyservice-1  log.error_count     30
13        currencyservice-1    log.row_count     30
14        currencyservice-2    log.row_count     30
15        currencyservice-2  log.error_count     30
16        currencyservice-0    log.row_count     30
17        currencyservice-0  log.error_count     30
18  recommendationservice-1  log.error_count     30
19  recommendationservice-1    log.row_count     30,    total_rows        min_timestamp_utc8        max_timestamp_utc8  unique_cmdb_id_count
0         527 2022-03-21 00:30:01+08:00 2022-03-21 00:59:58+08:00                     3,       cmdb_id  count
0  frontend-0    222
1  frontend-1    158
2  frontend-2    147,               timestamp_utc8     cmdb_id                                            message
0  2022-03-21 00:30:01+08:00  frontend-1  severity: warning, message: failed to retrieve...
1  2022-03-21 00:30:02+08:00  frontend-2  severity: warning, message: failed to retrieve...
2  2022-03-21 00:30:10+08:00  frontend-1  severity: warning, message: failed to retrieve...
3  2022-03-21 00:30:15+08:00  frontend-2  severity: warning, message: failed to retrieve...
4  2022-03-21 00:30:17+08:00  frontend-0  severity: warning, message: failed to retrieve...
5  2022-03-21 00:30:19+08:00  frontend-1  severity: warning, message: failed to retrieve...
6  2022-03-21 00:30:25+08:00  frontend-2  severity: warning, message: failed to retrieve...
7  2022-03-21 00:30:26+08:00  frontend-1  severity: warning, message: failed to retrieve...
8  2022-03-21 00:30:34+08:00  frontend-0  severity: warning, message: failed to retrieve...
9  2022-03-21 00:30:34+08:00  frontend-2  severity: warning, message: failed to retrieve...
10 2022-03-21 00:30:41+08:00  frontend-1  severity: warning, message: failed to retrieve...
11 2022-03-21 00:30:45+08:00  frontend-0  severity: warning, message: failed to retrieve...
12 2022-03-21 00:30:48+08:00  frontend-1  severity: warning, message: failed to retrieve...
13 2022-03-21 00:30:51+08:00  frontend-2  severity: warning, message: failed to retrieve...
14 2022-03-21 00:30:53+08:00  frontend-0  severity: warning, message: failed to retrieve...
15 2022-03-21 00:31:00+08:00  frontend-1  severity: warning, message: failed to retrieve...
16 2022-03-21 00:31:03+08:00  frontend-0  severity: warning, message: failed to retrieve...
17 2022-03-21 00:31:07+08:00  frontend-2  severity: warning, message: failed to retrieve...
18 2022-03-21 00:31:08+08:00  frontend-0  severity: warning, message: failed to retrieve...
19 2022-03-21 00:31:12+08:00  frontend-0  severity: warning, message: failed to retrieve...)```
```

In [ ]:
# In[2]:

# Reuse metric_df already loaded in the kernel.
import pandas as pd

# Target cmdb_ids
targets = ["frontend-0","frontend-1","frontend-2","frontend2-0"]

# Ensure 'value' is numeric (coerce if needed)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Filter for target cmdb_ids (use full series in file as requested)
metric_filtered = metric_df[metric_df['cmdb_id'].isin(targets)].copy()

# Define quantile functions
def p50(x):
    return x.quantile(0.5)

def p95(x):
    return x.quantile(0.95)

# Aggregate per (cmdb_id, kpi_name)
metric_kpi_agg = (
    metric_filtered
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(count='count', min='min', p50=p50, p95=p95, max='max')
    .reset_index()
)

# Prepare compact top-20 table ordered by count
metric_kpi_agg_top20 = (
    metric_kpi_agg
    .sort_values('count', ascending=False)
    .loc[:, ['cmdb_id', 'kpi_name', 'count', 'p50', 'p95', 'min', 'max']]
    .head(20)
)

# Display top 20 and size of full aggregated table (stored in metric_kpi_agg)
metric_kpi_agg_top20, metric_kpi_agg.shape

```
Out[2]:
```
# Compose a concise plain-English summary based on the aggregated results already in kernel.
# Reuse metric_kpi_agg and metric_kpi_agg_top20 computed earlier.

# Ensure variables exist; if not, create safe placeholders
try:
    total_rows_agg = metric_kpi_agg.shape[0]
except NameError:
    metric_kpi_agg = pd.DataFrame()  # placeholder
    total_rows_agg = 0

try:
    top20 = metric_kpi_agg_top20.copy()
except NameError:
    top20 = metric_kpi_agg.head(20)

# Helper to extract a metric row summary
def row_summary(cmdb, kpi):
    df = metric_kpi_agg[(metric_kpi_agg['cmdb_id']==cmdb) & (metric_kpi_agg['kpi_name']==kpi)]
    if df.empty:
        return None
    r = df.iloc[0]
    return f"p50={r['p50']:.6g}, p95={r['p95']:.6g}, min={r['min']:.6g}, max={r['max']:.6g}, count={int(r['count'])}"

# Notable KPIs to call out (if present)
notables = {
    "frontend-0 container_cpu_usage_seconds": row_summary("frontend-0", "container.node-5.container_cpu_usage_seconds"),
    "frontend-0 container_cpu_system_seconds": row_summary("frontend-0", "container.node-5.container_cpu_system_seconds"),
    "frontend-0 file_descriptors": row_summary("frontend-0", "container.node-5.container_file_descriptors"),
    "frontend2-0 mesh source istio_response_metric": row_summary("frontend2-0", "mesh.source.frontend2.unknown.istio_response_mxlatency_ms_p50")
}

# Fallback: try to detect the mesh.source metric that appeared in top20 (observed in earlier run)
if notables["frontend2-0 mesh source istio_response_metric"] is None:
    # search for any mesh.source.* entry for frontend2-0 in the aggregated table
    mask = (metric_kpi_agg['cmdb_id']=='frontend2-0') & metric_kpi_agg['kpi_name'].str.contains('mesh.source', na=False)
    if mask.any():
        r = metric_kpi_agg[mask].iloc[0]
        notables["frontend2-0 mesh source istio_response_metric"] = f"p50={r['p50']:.6g}, p95={r['p95']:.6g}, min={r['min']:.6g}, max={r['max']:.6g}, count={int(r['count'])}"

# Build the summary text
summary_lines = [
    f"Aggregated per-(cmdb_id,kpi_name) for frontends (frontend-0, frontend-1, frontend-2, frontend2-0):",
    f"- Total distinct (cmdb_id,kpi_name) rows: {total_rows_agg}",
    f"- Top 20 table (stored as 'metric_kpi_agg_top20') shows the highest-count KPI groups (counts are typically 30 samples in this dataset).",
    "- Notable observed metrics (if present):"
]

for k,v in notables.items():
    summary_lines.append(f"  * {k}: {v or 'not present in aggregated table'}")

summary_lines.append("- Many IO/fs-related and some container metrics have zero values in this period; CPU usage metrics for frontend-0 show modest values (roughly 0.15–0.29 range for observed samples).")
summary_lines.append("- The full aggregated table is stored in kernel variable 'metric_kpi_agg' for later steps (shape: {}).".format(metric_kpi_agg.shape))

summary = "\n".join(summary_lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name  count            p50            p95            min            max
633  frontend2-0  mesh.source.frontend2.unknown.istio_response_m...     30       0.000000       0.000000       0.000000       0.000000
0     frontend-0         container.node-5.container_cpu_cfs_periods     30      91.500000     108.825000      70.000000     114.500000
1     frontend-0  container.node-5.container_cpu_cfs_throttled_p...     30       0.000000       0.000000       0.000000       0.000000
2     frontend-0  container.node-5.container_cpu_cfs_throttled_s...     30       0.000000       0.000000       0.000000       0.000000
3     frontend-0    container.node-5.container_cpu_load_average_10s     30       0.000000       0.000000       0.000000       0.000000
4     frontend-0      container.node-5.container_cpu_system_seconds     30       0.065000       0.090000       0.040000       0.095000
5     frontend-0       container.node-5.container_cpu_usage_seconds     30       0.240857       0.277012       0.150533       0.289966
6     frontend-0        container.node-5.container_cpu_user_seconds     30       0.130000       0.160000       0.055000       0.185000
7     frontend-0        container.node-5.container_file_descriptors     30      18.250000      19.000000      18.000000      19.000000
8     frontend-0     container.node-5.container_fs_inodes./dev/vda1     30       0.000000       0.000000       0.000000       0.000000
9     frontend-0  container.node-5.container_fs_inodes_free./dev...     30       0.000000       0.000000       0.000000       0.000000
10    frontend-0  container.node-5.container_fs_io_current./dev/...     30       0.000000       0.000000       0.000000       0.000000
11    frontend-0  container.node-5.container_fs_io_time_seconds....     30       0.000000       0.000000       0.000000       0.000000
12    frontend-0  container.node-5.container_fs_io_time_weighted...     30       0.000000       0.000000       0.000000       0.000000
13    frontend-0   container.node-5.container_fs_limit_MB./dev/vda1     30  604630.738281  604630.738281  604630.738281  604630.738281
14    frontend-0  container.node-5.container_fs_read_seconds./de...     30       0.000000       0.000000       0.000000       0.000000
15    frontend-0       container.node-5.container_fs_reads./dev/vda     30       0.000000       0.000000       0.000000       0.000000
16    frontend-0      container.node-5.container_fs_reads./dev/vda1     30       0.000000       0.000000       0.000000       0.000000
17    frontend-0    container.node-5.container_fs_reads_MB./dev/vda     30       0.000000       0.000000       0.000000       0.000000
18    frontend-0  container.node-5.container_fs_reads_merged./de...     30       0.000000       0.000000       0.000000       0.000000, (634, 7))```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables: metric_df, metric_kpi_agg
# Ensure required variables exist
assert 'metric_df' in globals(), "metric_df not found in kernel"
assert 'metric_kpi_agg' in globals(), "metric_kpi_agg not found in kernel"

# Ensure numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Target frontends
targets = ["frontend-0","frontend-1","frontend-2","frontend2-0"]

# Compute epoch seconds for window boundaries (UTC+8 -> convert to UTC)
start_ts = int(pd.Timestamp("2022-03-21 00:30:00", tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp("2022-03-21 00:59:00", tz=tz).tz_convert('UTC').timestamp())

# Filter metric records for targets within the incident window
window_df = metric_df[
    metric_df['cmdb_id'].isin(targets) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# Prepare global thresholds table (do NOT recompute from window; use metric_kpi_agg)
metric_kpi_global = metric_kpi_agg.rename(columns={
    'p50': 'global_p50',
    'p95': 'global_p95',
    'min': 'global_min',
    'max': 'global_max',
    'count': 'global_count'
})

# Join window records with global stats by (cmdb_id, kpi_name)
merged = window_df.merge(metric_kpi_global, on=['cmdb_id', 'kpi_name'], how='left')

# Define anomalies: value > global_p95 (must handle missing p95)
anomalies = merged[ (merged['global_p95'].notna()) & (merged['value'] > merged['global_p95']) ].copy()

# Store full anomalies table for later inspection
metric_anomalies_frontends = anomalies.copy()

# If no anomalies found, produce empty summary DataFrame with required columns
if anomalies.empty:
    summary_cols = [
        'cmdb_id','kpi_name','anomalies_count_in_window','earliest_anomaly_timestamp_utc8',
        'max_value_in_window','global_p50','global_p95','global_min','global_max','severity_ratio'
    ]
    metric_anomalies_summary = pd.DataFrame(columns=summary_cols)
else:
    # Aggregate per (cmdb_id, kpi_name)
    agg = (
        anomalies
        .groupby(['cmdb_id','kpi_name'], dropna=False)
        .agg(
            anomalies_count_in_window = ('value','count'),
            earliest_ts = ('timestamp','min'),
            max_value_in_window = ('value','max'),
            global_p50 = ('global_p50','first'),
            global_p95 = ('global_p95','first'),
            global_min = ('global_min','first'),
            global_max = ('global_max','first')
        )
        .reset_index()
    )

    # Convert earliest_ts to UTC+8 datetime strings
    agg['earliest_anomaly_timestamp_utc8'] = pd.to_datetime(agg['earliest_ts'], unit='s', utc=True).dt.tz_convert(tz)

    # Compute severity_ratio (null if global_p95 == 0 or NaN)
    agg['severity_ratio'] = np.where(
        (agg['global_p95'].notna()) & (agg['global_p95'] != 0),
        agg['max_value_in_window'] / agg['global_p95'],
        np.nan
    )

    # Select and order columns
    metric_anomalies_summary = agg[
        ['cmdb_id','kpi_name','anomalies_count_in_window','earliest_anomaly_timestamp_utc8',
         'max_value_in_window','global_p50','global_p95','global_min','global_max','severity_ratio']
    ].copy()

    # Order by severity_ratio desc, then by anomalies_count_in_window desc
    metric_anomalies_summary = metric_anomalies_summary.sort_values(
        by=['severity_ratio','anomalies_count_in_window'], ascending=[False, False]
    ).head(20)

# Final displayed variable: compact summary (max 20 rows)
metric_anomalies_summary

```
Out[3]:
```
Summary of metric anomalies for the four frontends during 2022-03-21 00:30–00:59 (UTC+8):

- Affected components: anomalies were detected across all four frontends (frontend-0, frontend-1, frontend-2, frontend2-0).  
- Highest-severity issues (top hits):
  - frontend-2: large spikes in mesh request metrics toward currencyservice (mesh.source.frontend.currencyservice.istio_request…) around 00:58 — max ≈ 51.6 vs global p95 ≈ 3.33 (severity_ratio ≈ 15.5).  
  - frontend2-0: elevated filesystem read/MB and related fs read counts around 00:40 (severity ratios ≈ 14.2 and ≈5.45).  
  - frontend-2: mesh.destination.* istio_request metrics spiking at ~00:58 (severity_ratio ≈ 11.6).  
  - frontend-0 and frontend-1: mesh related request/destination metrics also spike around 00:58 (severity ratios ~7–3).  
- Other notable resource anomalies:
  - CPU throttling-related KPIs for frontend-2 and frontend2-0 showed spikes (e.g., container_cpu_cfs_throttled_*), indicating throttling events (severity ratios up to ~11).  
  - frontend-2 showed a large jump in container_cpu_user_seconds at ~00:47 (severity_ratio ≈ 4.9).  
  - Some very large outliers in container memory or fs metrics were observed for frontend-2 (one entry with very large max value ~121,474 vs global p95 ~812 — severity_ratio ≈1.64).
- Timing pattern: many mesh-related anomalies cluster at ~00:58, while some I/O and memory related anomalies for frontend2-0 occur earlier (~00:33–00:40) and CPU/throttling spikes around ~00:50.
- Interpretation / next steps:
  - The frontends experienced significant service-mesh-level spikes (high latency/error metrics) to upstream services (notably currencyservice/unknown destinations) around 00:58 — these align with frontend error log activity and are likely the immediate cause of user-visible failures.  
  - Also investigate node/container resource issues for frontend2-0 (fs reads, memory failures) and CPU-throttling on frontend-2/frontends around the times shown — resource constraints could be contributing or independent issues.  
  - Recommended actions: inspect frontend pod logs/traces for ~00:58 (mesh call failures/timeouts), check upstream service (currencyservice) health and latency, and verify node/container resource metrics (CPU throttling, disk I/O, memory) for the frontends and their nodes (node-5/node-6 references appear in KPI names).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomalies_count_in_window earliest_anomaly_timestamp_utc8  max_value_in_window  global_p50    global_p95  global_min     global_max  severity_ratio
194   frontend-2  mesh.source.frontend.currencyservice.istio_req...                          2       2022-03-21 00:58:00+08:00            51.565625    0.070775      3.328424    0.044575      51.565625       15.492506
228  frontend2-0    container.node-6.container_fs_reads_MB./dev/vda                          2       2022-03-21 00:40:00+08:00             0.183594    0.000000      0.012891    0.000000       0.183594       14.242424
165   frontend-2  mesh.destination.unknown.frontend.istio_reques...                          2       2022-03-21 00:58:00+08:00            51.842750    0.980875      4.452100    0.844000      51.842750       11.644561
139   frontend-2  container.node-5.container_cpu_cfs_throttled_s...                          1       2022-03-21 00:50:00+08:00            14.027469    0.000000      1.266240    0.000000      14.027469       11.078051
43    frontend-0  mesh.source.frontend.currencyservice.istio_req...                          2       2022-03-21 00:58:00+08:00            49.095825    0.106162      5.231476    0.077700      49.095825        9.384698
17    frontend-0  mesh.destination.unknown.frontend.istio_reques...                          2       2022-03-21 00:58:00+08:00            50.502250    1.489625      6.923875    1.308750      50.502250        7.293929
233  frontend2-0  container.node-6.container_memory_failures.con...                          2       2022-03-21 00:40:00+08:00             2.000000    0.000000      0.366667    0.000000       2.000000        5.454545
235  frontend2-0  container.node-6.container_memory_failures.hie...                          2       2022-03-21 00:40:00+08:00             2.000000    0.000000      0.366667    0.000000       2.000000        5.454545
227  frontend2-0       container.node-6.container_fs_reads./dev/vda                          2       2022-03-21 00:40:00+08:00             6.000000    0.000000      1.100000    0.000000       6.000000        5.454545
158   frontend-2  container.node-5.container_network_receive_MB....                          2       2022-03-21 00:32:00+08:00             0.646356    0.110777      0.130872    0.039846       0.646356        4.938847
142   frontend-2        container.node-5.container_cpu_user_seconds                          2       2022-03-21 00:47:00+08:00             1.985000    0.090000      0.407750    0.025000       1.985000        4.868179
110   frontend-1  mesh.source.frontend.currencyservice.istio_req...                          2       2022-03-21 00:58:00+08:00             8.968900    0.072387      2.512870    0.047225       8.968900        3.569186
138   frontend-2  container.node-5.container_cpu_cfs_throttled_p...                          1       2022-03-21 00:50:00+08:00            56.000000    0.000000     16.500000    0.000000      56.000000        3.393939
87    frontend-1  mesh.destination.unknown.frontend.istio_reques...                          2       2022-03-21 00:58:00+08:00             9.102000    1.042125      3.647937    0.897750       9.102000        2.495109
223  frontend2-0  container.node-6.container_cpu_cfs_throttled_s...                          2       2022-03-21 00:33:00+08:00             0.081887    0.001250      0.036406    0.000000       0.081887        2.249246
103   frontend-1  mesh.source.frontend.cartservice.istio_request...                          2       2022-03-21 00:37:00+08:00             0.074425    0.026675      0.034254    0.020425       0.074425        2.172755
88    frontend-1  mesh.destination.unknown.frontend.istio_reques...                          2       2022-03-21 00:31:00+08:00             0.072575    0.020425      0.036511    0.000000       0.072575        1.987744
143   frontend-2     container.node-5.container_fs_inodes./dev/vda1                          2       2022-03-21 00:47:00+08:00             2.000000    0.000000      1.100000    0.000000       2.000000        1.818182
240  frontend2-0  container.node-6.container_network_receive_MB....                          2       2022-03-21 00:48:00+08:00             0.501235    0.162698      0.278805    0.107632       0.501235        1.797797
151   frontend-2  container.node-5.container_memory_failures.con...                          2       2022-03-21 00:48:00+08:00        121474.500000  812.250000  73974.925000  284.500000  121474.500000        1.642104```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing trace_df loaded earlier
targets = ["frontend-0","frontend-1","frontend-2","frontend2-0"]

# Ensure numeric values
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Filter for target frontends (use the full available series as requested)
trace_filtered = trace_df[trace_df['cmdb_id'].isin(targets)].copy()

# Aggregate per (cmdb_id, trace_name) using full series
trace_kpi_agg_frontends = (
    trace_filtered
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(
        count='count',
        min='min',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95),
        max='max'
    )
    .reset_index()
)

# Prepare compact top-20 ordered by count
trace_kpi_agg_frontends_top20 = (
    trace_kpi_agg_frontends
    .sort_values('count', ascending=False)
    .loc[:, ['cmdb_id','trace_name','count','p50','p95','min','max']]
    .head(20)
)

# Display compact result (top20) and shape of full aggregated table
trace_kpi_agg_frontends_top20, trace_kpi_agg_frontends.shape

```
Out[4]:
```
summary = (
    "Trace aggregation summary for frontends (full series):\n"
    "- Total distinct (cmdb_id, trace_name) pairs: 312 (stored in variable 'trace_kpi_agg_frontends').\n"
    "- Top observations (examples from the top-20 by count, each group has 30 samples):\n"
    "  * frontend-0: trace.from_root.duration_mean — p50=0.0407s, p95=0.8981s, min=0.0346s, max=2.2815s.\n"
    "  * frontend-0: trace.from_root.duration_p95 — p50=0.0568s, p95=4.1589s (shows some higher tail latency).\n"
    "  * frontend-0: trace.self.row_count — p50=372, p95=442.3 (high per-minute request counts on the frontend).\n"
    "  * Many trace.to_<service>.* duration metrics (e.g., to adservice) are extremely small (microsecond range: p50≈1e-05–1e-04s, p95 still very small), indicating low downstream call latency for those services.\n"
    "  * frontend2-0: trace.to_recommendationservice2-0.row_count — p50=27.5, p95=33.55 (notable per-minute traffic to recommendationservice2-0).\n"
    "  * Error-rate traces in the top groups are typically 0 (no widespread trace.error_rate spikes among top pairs).\n"
    "- Interpretation: frontends serve substantial request volumes (high row_count), most downstream RPC latencies are very low, but there are measurable tail latencies in the front-end-level 'from_root' traces (p95 up to several seconds) which merit inspection. Use 'trace_kpi_agg_frontends' for detailed per-trace analysis."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                   trace_name  count         p50         p95         min         max
307  frontend2-0  trace.to_recommendationservice2-0.row_count     30   27.500000   33.550000   19.000000   40.000000
0     frontend-0                trace.from_root.duration_mean     30    0.040729    0.898057    0.034600    2.281510
1     frontend-0                 trace.from_root.duration_p95     30    0.056819    4.158869    0.050499   10.512334
2     frontend-0                   trace.from_root.error_rate     30    0.000000    0.000000    0.000000    0.000000
3     frontend-0                    trace.from_root.row_count     30   38.000000   44.000000   20.000000   48.000000
4     frontend-0                     trace.self.duration_mean     30    0.004011    0.084247    0.003649    0.244228
5     frontend-0                      trace.self.duration_p95     30    0.006691    0.118570    0.006358    0.623405
6     frontend-0                        trace.self.error_rate     30    0.000000    0.000000    0.000000    0.000000
7     frontend-0                         trace.self.row_count     30  372.000000  442.300000  232.000000  472.000000
8     frontend-0           trace.to_adservice-0.duration_mean     30    0.000013    0.000017    0.000012    0.000216
9     frontend-0            trace.to_adservice-0.duration_p95     30    0.000016    0.000029    0.000014    0.001069
10    frontend-0              trace.to_adservice-0.error_rate     30    0.000000    0.000000    0.000000    0.000000
11    frontend-0               trace.to_adservice-0.row_count     30    7.500000   10.550000    4.000000   11.000000
12    frontend-0           trace.to_adservice-1.duration_mean     30    0.000012    0.000014    0.000011    0.000016
13    frontend-0            trace.to_adservice-1.duration_p95     30    0.000014    0.000020    0.000013    0.000031
14    frontend-0              trace.to_adservice-1.error_rate     30    0.000000    0.000000    0.000000    0.000000
15    frontend-0               trace.to_adservice-1.row_count     30    7.500000   10.000000    3.000000   12.000000
16    frontend-0           trace.to_adservice-2.duration_mean     30    0.000013    0.000015    0.000013    0.000015
17    frontend-0            trace.to_adservice-2.duration_p95     30    0.000015    0.000021    0.000013    0.000023
18    frontend-0              trace.to_adservice-2.error_rate     30    0.000000    0.000000    0.000000    0.000000, (312, 7))```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables in kernel: trace_df, trace_kpi_agg_frontends
tz = pytz.timezone('Asia/Shanghai')

# Ensure inputs exist
assert 'trace_df' in globals(), "trace_df not found in kernel"
assert 'trace_kpi_agg_frontends' in globals(), "trace_kpi_agg_frontends not found in kernel"

# Ensure numeric values
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Target frontends and window (UTC+8)
targets = ["frontend-0","frontend-1","frontend-2","frontend2-0"]
start_ts = int(pd.Timestamp("2022-03-21 00:30:00", tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp("2022-03-21 00:59:00", tz=tz).tz_convert('UTC').timestamp())

# Filter trace records within window for target frontends
trace_window = trace_df[
    trace_df['cmdb_id'].isin(targets) &
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# Prepare global aggregated thresholds (do NOT recompute from window)
trace_global = trace_kpi_agg_frontends.rename(columns={
    'p50': 'global_p50',
    'p95': 'global_p95',
    'min': 'global_min',
    'max': 'global_max',
    'count': 'global_count'
})

# Join window records with global stats by (cmdb_id, trace_name)
merged = trace_window.merge(trace_global, on=['cmdb_id','trace_name'], how='left')

# Define anomalies: value > global_p95 (and global_p95 not null)
trace_anomalies_frontends = merged[(merged['global_p95'].notna()) & (merged['value'] > merged['global_p95'])].copy()

# Build compact per-(cmdb_id, trace_name) summary for anomalies
if trace_anomalies_frontends.empty:
    cols = [
        'cmdb_id','trace_name','anomalies_count_in_window','earliest_anomaly_timestamp_utc8',
        'max_value_in_window','global_p50','global_p95','global_min','global_max','severity_ratio'
    ]
    trace_anomalies_summary = pd.DataFrame(columns=cols)
else:
    agg = (
        trace_anomalies_frontends
        .groupby(['cmdb_id','trace_name'], dropna=False)
        .agg(
            anomalies_count_in_window = ('value','count'),
            earliest_ts = ('timestamp','min'),
            max_value_in_window = ('value','max'),
            global_p50 = ('global_p50','first'),
            global_p95 = ('global_p95','first'),
            global_min = ('global_min','first'),
            global_max = ('global_max','first')
        )
        .reset_index()
    )

    # Convert earliest timestamp to UTC+8 tz-aware datetime
    agg['earliest_anomaly_timestamp_utc8'] = pd.to_datetime(agg['earliest_ts'], unit='s', utc=True).dt.tz_convert(tz)

    # severity_ratio (null if global_p95 == 0 or NaN)
    agg['severity_ratio'] = np.where(
        (agg['global_p95'].notna()) & (agg['global_p95'] != 0),
        agg['max_value_in_window'] / agg['global_p95'],
        np.nan
    )

    trace_anomalies_summary = agg[
        ['cmdb_id','trace_name','anomalies_count_in_window','earliest_anomaly_timestamp_utc8',
         'max_value_in_window','global_p50','global_p95','global_min','global_max','severity_ratio']
    ].sort_values(by=['severity_ratio','anomalies_count_in_window'], ascending=[False, False]).head(20)

# Final outputs: store full per-record anomalies in kernel variable (already named) and display compact summary
trace_anomalies_frontends, trace_anomalies_summary

```
Out[5]:
```
Summary of trace anomalies for the four frontends during 2022-03-21 00:30–00:59 (UTC+8):

- Overall: 375 anomalous trace records were found in the window (stored in kernel variable 'trace_anomalies_frontends'). The per-(cmdb_id,trace_name) summary (top 20 by severity) highlights multiple large tail-latency spikes and some high ratios for short-duration traces.

- Highest-severity anomalies (examples from top results):
  - frontend-0: trace.to_adservice-0.duration_p95 — max ≈ 0.001069s vs global p95 ≈ 0.000029s (severity ≈ 36.9) at 00:37.
  - frontend-1: trace.to_cartservice-2.duration_p95 — max ≈ 0.02145s vs global p95 ≈ 0.00065s (severity = 33.0) at 00:38.
  - frontend-2: trace.from_root.duration_p95 — max ≈ 46.50s vs global p95 ≈ 1.55s (severity ≈ 29.9) at 00:58 (very large frontend tail latency).
  - frontend-1: trace.self.duration_p95 — max ≈ 0.2119s vs global p95 ≈ 0.00712s (severity ≈ 29.7) at 00:58.
  - frontend-0: trace.from_root.duration_mean/p95 and trace.self.duration_p95 show notable spikes around 00:58 (severity ratios ~2.5–5.3).

- Timing pattern:
  - Many severe front-end level latency anomalies (from_root/self and destination traces) cluster at ~00:58.
  - Some earlier spikes targeting downstream services (adservice, cartservice) occur around 00:33–00:38.
  - A broad set of destination-duration anomalies (to various services) appear with high severity ratios, though absolute times for those vary.

- Interpretation and next steps:
  - The dominant signal is large frontend tail latencies (~00:58) that likely caused or coincide with user-facing errors. These should be investigated first (inspect frontend traces and logs around 00:58).
  - Also investigate the earlier spikes (~00:33–00:38) targeting adservice/cartservice — they may be related upstream latency events or isolated downstream slow calls.
  - Correlate these trace spikes with the metric anomalies and error_logs already observed (mesh request/response spikes and frontend warning logs) and with node/container resource metrics (CPU, throttling, I/O) for the frontend pods and their nodes.
  - For deeper analysis use the stored tables: 'trace_kpi_agg_frontends' (global baselines) and 'trace_anomalies_frontends' (per-record anomalies).

The original code execution output of IPython Kernel is also provided below for reference:

(       timestamp      cmdb_id                                trace_name     value  global_count  global_min  global_p50  global_p95  global_max
28    1647793800   frontend-0      trace.to_cartservice-2.duration_mean  0.000100            30    0.000000    0.000000    0.000092    0.000143
29    1647793800   frontend-0       trace.to_cartservice-2.duration_p95  0.000550            30    0.000000    0.000000    0.000505    0.000700
80    1647793800   frontend-0  trace.to_shippingservice-0.duration_mean  0.000072            27    0.000056    0.000058    0.000067    0.000072
81    1647793800   frontend-0   trace.to_shippingservice-0.duration_p95  0.000103            27    0.000056    0.000061    0.000080    0.000103
124   1647793800   frontend-1  trace.to_currencyservice-0.duration_mean  0.000143            30    0.000084    0.000098    0.000130    0.000144
...          ...          ...                                       ...       ...           ...         ...         ...         ...         ...
8417  1647795540   frontend-2              trace.from_root.duration_p95  2.764918            30    0.049604    0.057244    1.554699   46.504499
8420  1647795540   frontend-2                  trace.self.duration_mean  0.084009            30    0.003533    0.003992    0.048270    0.526141
8421  1647795540   frontend-2                   trace.self.duration_p95  0.007468            30    0.006371    0.006768    0.007289    0.007574
8436  1647795540   frontend-2      trace.to_cartservice-0.duration_mean  0.000250            30    0.000000    0.000000    0.000237    0.000250
8529  1647795540  frontend2-0  trace.to_shippingservice2-0.duration_p95  0.000227            29    0.000062    0.000071    0.000225    0.000228

[375 rows x 9 columns],         cmdb_id                                trace_name  anomalies_count_in_window earliest_anomaly_timestamp_utc8  max_value_in_window  global_p50  global_p95  global_min  global_max  severity_ratio
7    frontend-0         trace.to_adservice-0.duration_p95                          2       2022-03-21 00:37:00+08:00             0.001069    0.000016    0.000029    0.000014    0.001069       36.925734
84   frontend-1       trace.to_cartservice-2.duration_p95                          1       2022-03-21 00:38:00+08:00             0.021450    0.000000    0.000650    0.000000    0.021450       33.000000
83   frontend-1      trace.to_cartservice-2.duration_mean                          1       2022-03-21 00:38:00+08:00             0.003900    0.000000    0.000125    0.000000    0.003900       31.200000
129  frontend-2              trace.from_root.duration_p95                          2       2022-03-21 00:58:00+08:00            46.504499    0.057244    1.554699    0.049604   46.504499       29.912224
66   frontend-1                   trace.self.duration_p95                          2       2022-03-21 00:58:00+08:00             0.211886    0.006563    0.007124    0.006184    0.211886       29.742560
6    frontend-0        trace.to_adservice-0.duration_mean                          2       2022-03-21 00:37:00+08:00             0.000216    0.000013    0.000017    0.000012    0.000216       13.051360
131  frontend-2                  trace.self.duration_mean                          2       2022-03-21 00:58:00+08:00             0.526141    0.003992    0.048270    0.003533    0.526141       10.899857
128  frontend-2             trace.from_root.duration_mean                          2       2022-03-21 00:58:00+08:00             5.115351    0.040698    0.512289    0.035280    5.115351        9.985278
137  frontend-2         trace.to_adservice-1.duration_p95                          2       2022-03-21 00:33:00+08:00             0.000174    0.000014    0.000032    0.000012    0.000174        5.515055
4    frontend-0                   trace.self.duration_p95                          2       2022-03-21 00:58:00+08:00             0.623405    0.006691    0.118570    0.006358    0.623405        5.257696
25   frontend-0  trace.to_checkoutservice-1.duration_mean                          1       2022-03-21 00:58:00+08:00             1.246646    0.038687    0.425183    0.028629    1.246646        2.932024
26   frontend-0   trace.to_checkoutservice-1.duration_p95                          1       2022-03-21 00:58:00+08:00             1.246646    0.038687    0.425183    0.028629    1.246646        2.932024
3    frontend-0                  trace.self.duration_mean                          2       2022-03-21 00:58:00+08:00             0.244228    0.004011    0.084247    0.003649    0.244228        2.898940
145  frontend-2       trace.to_cartservice-1.duration_p95                          2       2022-03-21 00:32:00+08:00             0.002100    0.000000    0.000727    0.000000    0.002100        2.886598
144  frontend-2      trace.to_cartservice-1.duration_mean                          2       2022-03-21 00:32:00+08:00             0.000429    0.000000    0.000156    0.000000    0.000429        2.746479
136  frontend-2        trace.to_adservice-1.duration_mean                          2       2022-03-21 00:33:00+08:00             0.000053    0.000013    0.000019    0.000012    0.000053        2.731959
63   frontend-1              trace.from_root.duration_p95                          2       2022-03-21 00:58:00+08:00             1.959722    0.056716    0.744558    0.049951    1.959722        2.632062
0    frontend-0             trace.from_root.duration_mean                          2       2022-03-21 00:58:00+08:00             2.281510    0.040729    0.898057    0.034600    2.281510        2.540495
1    frontend-0              trace.from_root.duration_p95                          2       2022-03-21 00:58:00+08:00            10.512334    0.056819    4.158869    0.050499   10.512334        2.527690
123  frontend-1   trace.to_shippingservice-1.duration_p95                          2       2022-03-21 00:51:00+08:00             0.000272    0.000062    0.000108    0.000055    0.000272        2.516189)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure source anomalies table exists
assert 'trace_anomalies_frontends' in globals(), "trace_anomalies_frontends not found in kernel"

# Filter records where trace_name contains 'currencyservice' (case-sensitive as requested)
trace_anoms_frontend_to_currency = trace_anomalies_frontends[trace_anomalies_frontends['trace_name'].str.contains('currencyservice', na=False)].copy()

# If empty, prepare empty summary with required columns
if trace_anoms_frontend_to_currency.empty:
    cols = ['cmdb_id','trace_name','anomalies_count_in_window','earliest_anomaly_timestamp_utc8',
            'max_value_in_window','global_p95','severity_ratio']
    trace_anoms_frontend_to_currency_summary = pd.DataFrame(columns=cols)
    distinct_count = 0
    earliest_overall = pd.NaT
else:
    # Aggregate per (cmdb_id, trace_name)
    agg = (
        trace_anoms_frontend_to_currency
        .groupby(['cmdb_id','trace_name'], dropna=False)
        .agg(
            anomalies_count_in_window = ('value','count'),
            earliest_ts = ('timestamp','min'),
            max_value_in_window = ('value','max'),
            global_p95 = ('global_p95','first')
        )
        .reset_index()
    )

    # Convert earliest_ts to UTC+8 tz-aware datetime
    agg['earliest_anomaly_timestamp_utc8'] = pd.to_datetime(agg['earliest_ts'], unit='s', utc=True).dt.tz_convert(tz)

    # Compute severity_ratio (null if global_p95 == 0 or NaN)
    agg['severity_ratio'] = np.where(
        (agg['global_p95'].notna()) & (agg['global_p95'] != 0),
        agg['max_value_in_window'] / agg['global_p95'],
        np.nan
    )

    # Prepare final summary columns and order
    trace_anoms_frontend_to_currency_summary = agg[
        ['cmdb_id','trace_name','anomalies_count_in_window','earliest_anomaly_timestamp_utc8',
         'max_value_in_window','global_p95','severity_ratio']
    ].sort_values(by=['severity_ratio','anomalies_count_in_window'], ascending=[False, False]).head(20)

    distinct_count = trace_anoms_frontend_to_currency_summary.shape[0]
    earliest_overall = trace_anoms_frontend_to_currency_summary['earliest_anomaly_timestamp_utc8'].min()

# Display compact results: summary (<=20 rows), distinct count, earliest overall anomaly timestamp
trace_anoms_frontend_to_currency_summary, distinct_count, earliest_overall

```
Out[6]:
```
import pandas as pd

# Build a plain-English summary string from existing kernel variables produced earlier.
# Expecting these to exist: trace_anoms_frontend_to_currency (full filtered rows) and trace_anoms_frontend_to_currency_summary (aggregated summary)
assert 'trace_anoms_frontend_to_currency' in globals(), "trace_anoms_frontend_to_currency not found"
assert 'trace_anoms_frontend_to_currency_summary' in globals(), "trace_anoms_frontend_to_currency_summary not found"

df = trace_anoms_frontend_to_currency_summary.copy()

# Distinct anomalous groups and earliest timestamp
distinct_count = int(df.shape[0])
if distinct_count > 0:
    earliest_overall = df['earliest_anomaly_timestamp_utc8'].min()
    earliest_str = pd.to_datetime(earliest_overall).strftime('%Y-03-21 %H:%M:%S%z') if not pd.isna(earliest_overall) else str(earliest_overall)
else:
    earliest_str = "N/A"

# Top 3 by severity for concise callout
top3 = df.sort_values('severity_ratio', ascending=False).head(3)
top_lines = []
for _, r in top3.iterrows():
    cmdb = r['cmdb_id']
    tname = r['trace_name']
    ts = pd.to_datetime(r['earliest_anomaly_timestamp_utc8']).strftime('%Y-%m-%d %H:%M:%S%z')
    maxv = r['max_value_in_window']
    p95 = r['global_p95']
    sev = r['severity_ratio']
    top_lines.append(f"- {cmdb} | {tname} at {ts}: max={maxv} vs global_p95={p95} (severity={sev:.3f})")

summary = (
    f"Frontend→currencyservice trace anomalies (compact):\n"
    f"- Distinct frontend→currencyservice trace groups anomalous: {distinct_count}\n"
    f"- Earliest anomaly among them: {earliest_str} (UTC+8)\n"
    f"- Top anomalies by severity:\n"
    + ("\n".join(top_lines) if top_lines else "  (no anomalies found)") +
    "\n\nNotes:\n"
    "- Most groups show small absolute durations but exceed their historical p95 by modest factors (severity ~1.03–1.16).\n"
    "- A few early anomalies (around 00:30–00:38) show higher severity (e.g., severity ~2.1 and ~1.33), indicating short-lived elevated latency to currencyservice from some frontends.\n"
    "- The filtered full records are stored in kernel variable 'trace_anoms_frontend_to_currency' for further inspection."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                 trace_name  anomalies_count_in_window earliest_anomaly_timestamp_utc8  max_value_in_window  global_p95  severity_ratio
9    frontend-1    trace.to_currencyservice-0.duration_p95                          2       2022-03-21 00:30:00+08:00             0.000320    0.000152        2.105263
12   frontend-1    trace.to_currencyservice-1.duration_p95                          2       2022-03-21 00:30:00+08:00             0.000307    0.000230        1.332176
26  frontend2-0  trace.to_currencyservice2-0.duration_mean                          2       2022-03-21 00:33:00+08:00             0.002917    0.002513        1.160833
14   frontend-1   trace.to_currencyservice-2.duration_mean                          2       2022-03-21 00:49:00+08:00             0.000121    0.000105        1.157341
27  frontend2-0   trace.to_currencyservice2-0.duration_p95                          2       2022-03-21 00:33:00+08:00             0.010802    0.009363        1.153696
17   frontend-2   trace.to_currencyservice-0.duration_mean                          2       2022-03-21 00:39:00+08:00             0.000125    0.000112        1.119570
5    frontend-0   trace.to_currencyservice-2.duration_mean                          1       2022-03-21 00:47:00+08:00             0.000126    0.000113        1.115044
8    frontend-1   trace.to_currencyservice-0.duration_mean                          2       2022-03-21 00:30:00+08:00             0.000144    0.000130        1.104294
22   frontend-2       trace.to_currencyservice-1.row_count                          2       2022-03-21 00:51:00+08:00            36.000000   32.650000        1.102603
3    frontend-0    trace.to_currencyservice-1.duration_p95                          2       2022-03-21 00:33:00+08:00             0.000145    0.000134        1.084112
10   frontend-1       trace.to_currencyservice-0.row_count                          2       2022-03-21 00:37:00+08:00            41.000000   38.100000        1.076115
13   frontend-1       trace.to_currencyservice-1.row_count                          2       2022-03-21 00:37:00+08:00            41.000000   38.100000        1.076115
16   frontend-1       trace.to_currencyservice-2.row_count                          2       2022-03-21 00:37:00+08:00            41.000000   38.200000        1.073298
25   frontend-2       trace.to_currencyservice-2.row_count                          2       2022-03-21 00:51:00+08:00            35.000000   32.650000        1.071975
7    frontend-0       trace.to_currencyservice-2.row_count                          2       2022-03-21 00:34:00+08:00            56.000000   52.550000        1.065652
1    frontend-0       trace.to_currencyservice-0.row_count                          2       2022-03-21 00:34:00+08:00            56.000000   52.650000        1.063628
19   frontend-2       trace.to_currencyservice-0.row_count                          2       2022-03-21 00:51:00+08:00            35.000000   33.100000        1.057402
20   frontend-2   trace.to_currencyservice-1.duration_mean                          2       2022-03-21 00:38:00+08:00             0.000117    0.000113        1.035857
4    frontend-0       trace.to_currencyservice-1.row_count                          2       2022-03-21 00:34:00+08:00            55.000000   53.100000        1.035782
6    frontend-0    trace.to_currencyservice-2.duration_p95                          2       2022-03-21 00:32:00+08:00             0.000136    0.000131        1.035008, 20, Timestamp('2022-03-21 00:30:00+0800', tz='Asia/Shanghai'))```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df from kernel
assert 'metric_df' in globals(), "metric_df not found in kernel"

# Ensure numeric 'value'
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Targets: currencyservice pods
targets = ["currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0"]

# 1) Aggregate full-series per (cmdb_id, kpi_name) for currencyservice pods (global thresholds must be from full series)
metric_kpi_agg_currency = (
    metric_df[metric_df['cmdb_id'].isin(targets)]
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .agg(
        count='count',
        global_min='min',
        global_p50=lambda x: x.quantile(0.5),
        global_p95=lambda x: x.quantile(0.95),
        global_max='max'
    )
    .reset_index()
)

# 2) Detect anomalies in incident window by joining window records to global p95
start_ts = int(pd.Timestamp("2022-03-21 00:30:00", tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp("2022-03-21 00:59:00", tz=tz).tz_convert('UTC').timestamp())

# Filter window records for the targets
metric_window_currency = metric_df[
    metric_df['cmdb_id'].isin(targets) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# Join with global aggregated thresholds
metric_window_joined = metric_window_currency.merge(
    metric_kpi_agg_currency,
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Define anomalies: value > global_p95 (and global_p95 not null)
metric_anoms_currency = metric_window_joined[
    metric_window_joined['global_p95'].notna() &
    (metric_window_joined['value'] > metric_window_joined['global_p95'])
].copy()

# Store full anomalies table in kernel variable (as requested)
# (variable name: metric_anoms_currency)
# Now build compact per-(cmdb_id,kpi_name) summary
if metric_anoms_currency.empty:
    cols = [
        'cmdb_id','kpi_name','anomalies_count_in_window','earliest_anomaly_timestamp_utc8',
        'max_value_in_window','global_p50','global_p95','global_min','global_max','severity_ratio'
    ]
    metric_currency_anoms_summary = pd.DataFrame(columns=cols)
else:
    agg = (
        metric_anoms_currency
        .groupby(['cmdb_id','kpi_name'], dropna=False)
        .agg(
            anomalies_count_in_window = ('value','count'),
            earliest_ts = ('timestamp','min'),
            max_value_in_window = ('value','max'),
            global_p50 = ('global_p50','first'),
            global_p95 = ('global_p95','first'),
            global_min = ('global_min','first'),
            global_max = ('global_max','first')
        )
        .reset_index()
    )

    # Convert earliest timestamp to UTC+8 tz-aware datetime
    agg['earliest_anomaly_timestamp_utc8'] = pd.to_datetime(agg['earliest_ts'], unit='s', utc=True).dt.tz_convert(tz)

    # severity_ratio: null if global_p95 is 0 or NaN
    agg['severity_ratio'] = np.where(
        (agg['global_p95'].notna()) & (agg['global_p95'] != 0),
        agg['max_value_in_window'] / agg['global_p95'],
        np.nan
    )

    metric_currency_anoms_summary = agg[
        ['cmdb_id','kpi_name','anomalies_count_in_window','earliest_anomaly_timestamp_utc8',
         'max_value_in_window','global_p50','global_p95','global_min','global_max','severity_ratio']
    ].sort_values(by=['severity_ratio','anomalies_count_in_window'], ascending=[False, False]).head(20)

# The aggregated currency metrics table and anomalies table are stored in:
# - metric_kpi_agg_currency
# - metric_anoms_currency

# Final compact summary output (max 20 rows)
metric_currency_anoms_summary

```
Out[7]:
```
summary = (
    "Currencyservice metric anomalies (2022-03-21 00:30–00:59 UTC+8) — concise findings:\n"
    "- Anomalies were detected across all currencyservice pods (currencyservice-0/1/2 and currencyservice2-0), typically as short bursts (most groups have 2 anomalous samples).\n"
    "- Highest-severity signals:\n"
    "  * currencyservice-0: large spike in network receive MB at ~00:48 (severity ≈ 8.6) — big relative jump vs historical p95.\n"
    "  * currencyservice-0: mesh destination metric (frontend→currencyservice) spike at ~00:39 (severity ≈ 6.3).\n"
    "- Memory-failure metrics are elevated on currencyservice-1 and currencyservice-0 (~00:38 and ~00:47) with high absolute counts (thousands) and severity ~1.67–1.76 — indicates container memory failure events.\n"
    "- Other observations:\n"
    "  * network_receive_MB spikes also seen on currencyservice2-0 and currencyservice-2 (severity ~1.4–1.6) around 00:37–00:48.\n"
    "  * currencyservice2-0 shows CPU throttling and very large memory_mapped_file values at ~00:30 (severity ~1.33), suggesting resource pressure on node-6.\n"
    "- Pattern: a mix of network and memory-related anomalies with some mesh-related latency/traffic spikes. Many anomalies are brief but several point to resource issues (memory failures, throttling) on currencyservice pods or their nodes.\n"
    "- Recommended next steps: inspect currencyservice pod logs and container runtimes around the noted timestamps, check node-level metrics for node-5/node-6 (memory, disk, network, CPU throttling), and correlate with frontend trace/metric anomalies to determine if currencyservice issues contributed to frontend errors."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomalies_count_in_window earliest_anomaly_timestamp_utc8  max_value_in_window     global_p50     global_p95    global_min    global_max  severity_ratio
12    currencyservice-0  container.node-5.container_network_receive_MB....                          2       2022-03-21 00:48:00+08:00         5.848460e-01       0.060321       0.067690      0.045203  5.848460e-01        8.640012
23    currencyservice-0  mesh.destination.frontend.currencyservice.isti...                          2       2022-03-21 00:39:00+08:00         8.612500e-02       0.007875       0.013647      0.002625  8.612500e-02        6.310680
34    currencyservice-1  container.node-5.container_memory_failures.con...                          2       2022-03-21 00:38:00+08:00         1.650000e+03     743.750000     936.175000    451.500000  1.650000e+03        1.762491
35    currencyservice-1  container.node-5.container_memory_failures.hie...                          2       2022-03-21 00:38:00+08:00         1.650000e+03     743.750000     936.175000    451.500000  1.650000e+03        1.762491
7     currencyservice-0  container.node-5.container_memory_failures.con...                          2       2022-03-21 00:47:00+08:00         1.618000e+03     791.000000     970.250000    471.500000  1.618000e+03        1.667611
8     currencyservice-0  container.node-5.container_memory_failures.hie...                          2       2022-03-21 00:47:00+08:00         1.618000e+03     791.000000     970.250000    471.500000  1.618000e+03        1.667611
102  currencyservice2-0  container.node-6.container_network_receive_MB....                          2       2022-03-21 00:48:00+08:00         3.553740e-01       0.073679       0.218142      0.034768  3.553740e-01        1.629094
66    currencyservice-2  container.node-5.container_network_receive_MB....                          2       2022-03-21 00:37:00+08:00         3.344641e-01       0.059874       0.212737      0.043678  3.344641e-01        1.572195
39    currencyservice-1  container.node-5.container_network_receive_MB....                          2       2022-03-21 00:53:00+08:00         3.280339e-01       0.058192       0.209813      0.036841  3.280339e-01        1.563457
71    currencyservice-2  mesh.destination.checkoutservice.currencyservi...                          2       2022-03-21 00:51:00+08:00         1.575000e-03       0.000000       0.001050      0.000000  1.575000e-03        1.500000
29    currencyservice-1      container.node-5.container_cpu_system_seconds                          1       2022-03-21 00:45:00+08:00         3.000000e-02       0.015000       0.020000      0.010000  3.000000e-02        1.500000
44    currencyservice-1  mesh.destination.checkoutservice.currencyservi...                          2       2022-03-21 00:43:00+08:00         1.575000e-03       0.000525       0.001050      0.000000  1.575000e-03        1.500000
61    currencyservice-2  container.node-5.container_memory_failures.con...                          2       2022-03-21 00:35:00+08:00         1.319500e+03     758.250000     914.850000    548.000000  1.319500e+03        1.442313
62    currencyservice-2  container.node-5.container_memory_failures.hie...                          2       2022-03-21 00:35:00+08:00         1.319500e+03     758.250000     914.850000    548.000000  1.319500e+03        1.442313
77    currencyservice-2  mesh.destination.frontend.currencyservice.isti...                          2       2022-03-21 00:47:00+08:00         1.260000e-02       0.005250       0.008925      0.002625  1.260000e-02        1.411765
50    currencyservice-1  mesh.destination.frontend.currencyservice.isti...                          2       2022-03-21 00:47:00+08:00         2.097500e-02       0.008400       0.015304      0.003675  2.097500e-02        1.370579
83   currencyservice2-0  container.node-6.container_cpu_cfs_throttled_p...                          2       2022-03-21 00:33:00+08:00         8.000000e+00       3.000000       5.850000      0.333333  8.000000e+00        1.367521
84   currencyservice2-0  container.node-6.container_cpu_cfs_throttled_s...                          2       2022-03-21 00:32:00+08:00         2.993692e-01       0.040717       0.220330      0.001426  2.993692e-01        1.358732
98   currencyservice2-0      container.node-6.container_memory_mapped_file                          2       2022-03-21 00:30:00+08:00         1.171456e+06  110592.000000  881800.533333  12288.000000  1.171456e+06        1.328482
87   currencyservice2-0        container.node-6.container_cpu_user_seconds                          2       2022-03-21 00:36:00+08:00         1.816667e-01       0.122500       0.143667      0.065000  1.816667e-01        1.264501```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_df loaded in kernel
assert 'trace_df' in globals(), "trace_df not found in kernel"

# Ensure numeric values
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Targets: currencyservice pods
targets = ["currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0"]

# 1) Aggregate full-series per (cmdb_id, trace_name) for currencyservice pods (global stats)
trace_kpi_agg_currency = (
    trace_df[trace_df['cmdb_id'].isin(targets)]
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(
        count='count',
        global_min='min',
        global_p50=lambda x: x.quantile(0.5),
        global_p95=lambda x: x.quantile(0.95),
        global_max='max'
    )
    .reset_index()
)

# 2) Detect anomalies in incident window by joining window records to global p95
start_ts = int(pd.Timestamp("2022-03-21 00:30:00", tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp("2022-03-21 00:59:00", tz=tz).tz_convert('UTC').timestamp())

trace_window_currency = trace_df[
    trace_df['cmdb_id'].isin(targets) &
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# Join with global aggregated baselines
trace_window_joined = trace_window_currency.merge(
    trace_kpi_agg_currency,
    on=['cmdb_id','trace_name'],
    how='left'
)

# Flag anomalies: value > global_p95 (and global_p95 not null)
trace_anoms_currency = trace_window_joined[
    trace_window_joined['global_p95'].notna() &
    (trace_window_joined['value'] > trace_window_joined['global_p95'])
].copy()

# 3) Build compact per-(cmdb_id, trace_name) summary of anomalies
if trace_anoms_currency.empty:
    cols = [
        'cmdb_id','trace_name','anomalies_count_in_window','earliest_anomaly_timestamp_utc8',
        'max_value_in_window','global_p50','global_p95','global_min','global_max','severity_ratio'
    ]
    trace_anoms_currency_summary = pd.DataFrame(columns=cols)
else:
    agg = (
        trace_anoms_currency
        .groupby(['cmdb_id','trace_name'], dropna=False)
        .agg(
            anomalies_count_in_window = ('value','count'),
            earliest_ts = ('timestamp','min'),
            max_value_in_window = ('value','max'),
            global_p50 = ('global_p50','first'),
            global_p95 = ('global_p95','first'),
            global_min = ('global_min','first'),
            global_max = ('global_max','first')
        )
        .reset_index()
    )

    agg['earliest_anomaly_timestamp_utc8'] = pd.to_datetime(agg['earliest_ts'], unit='s', utc=True).dt.tz_convert(tz)

    agg['severity_ratio'] = np.where(
        (agg['global_p95'].notna()) & (agg['global_p95'] != 0),
        agg['max_value_in_window'] / agg['global_p95'],
        np.nan
    )

    trace_anoms_currency_summary = agg[
        ['cmdb_id','trace_name','anomalies_count_in_window','earliest_anomaly_timestamp_utc8',
         'max_value_in_window','global_p50','global_p95','global_min','global_max','severity_ratio']
    ].sort_values(by=['severity_ratio','anomalies_count_in_window'], ascending=[False, False]).head(20)

# Final compact summary (top 20) returned for display; aggregated baselines and per-record anomalies stored as:
# - trace_kpi_agg_currency
# - trace_anoms_currency

trace_anoms_currency_summary

```
Out[8]:
```
Summary of trace anomalies for currencyservice pods during 2022-03-21 00:30–00:59 (UTC+8):

- Scope: anomalies were detected for multiple inbound trace types to currencyservice pods (from frontends and from checkoutservice), across currencyservice-0/1/2 and currencyservice2-0.

- Highest-severity group: currencyservice-0 — trace.from_frontend-1.duration_p95 spiked at 00:30 (max ≈ 0.000320 s vs global p95 ≈ 0.000152 s, severity ≈ 2.11).

- Other notable spikes:
  - Several trace.row_count groups (frontends → currencyservice) show modest increases (max row_count ≈ 41 vs p95 ≈ 38), severity ≈ 1.07–1.10, earliest around 00:34–00:37.
  - currencyservice-1 and -2 show anomalies for traces originating from checkoutservice (duration and row_count) with severity up to ~1.38, earliest around 00:31–00:33 and 00:44–00:48.
  - currencyservice2-0 shows elevated mean and p95 for traces from frontend2-0 (~00:33), severity ≈ 1.16.

- Timing pattern: the earliest currency-related trace anomalies appear at 00:30 (frontends → currencyservice), with additional groups around 00:33–00:38 and some around 00:44–00:51.

- Magnitude: most anomalies are small in absolute duration (sub-millisecond to milliseconds) but exceed historical tail (p95) by factors typically between ~1.03 and ~2.1. The largest relative spike is the p95-duration at 00:30 for currencyservice-0 (severity ≈ 2.11).

- Interpretation and next steps:
  - These anomalies indicate short-lived increases in latency and request counts to currencyservice rather than sustained large delays. The most concerning are the p95 spikes at ~00:30 and the checkoutservice→currencyservice increases around ~00:33–00:48.
  - Correlate these trace anomalies with the previously observed metric anomalies (memory/network/throttling) and frontend trace/metric spikes around 00:30–00:59.
  - Recommended actions: inspect currencyservice pod logs around 00:30–00:51, check checkoutservice and frontend traces for correlated errors/latency, and review node-level resource signals for the currencyservice pods (node-5/node-6).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                  trace_name  anomalies_count_in_window earliest_anomaly_timestamp_utc8  max_value_in_window  global_p50  global_p95  global_min  global_max  severity_ratio
11   currencyservice-0          trace.from_frontend-1.duration_p95                          2       2022-03-21 00:30:00+08:00             0.000320    0.000114    0.000152    0.000104    0.000320        2.105263
24   currencyservice-1      trace.from_checkoutservice-2.row_count                          1       2022-03-21 00:48:00+08:00             4.000000    1.000000    2.000000    1.000000    4.000000        2.000000
18   currencyservice-1      trace.from_checkoutservice-0.row_count                          2       2022-03-21 00:44:00+08:00             4.000000    1.000000    2.900000    1.000000    4.000000        1.379310
29   currencyservice-1          trace.from_frontend-1.duration_p95                          2       2022-03-21 00:30:00+08:00             0.000307    0.000113    0.000230    0.000101    0.000307        1.332176
22   currencyservice-1  trace.from_checkoutservice-2.duration_mean                          2       2022-03-21 00:33:00+08:00             0.000150    0.000104    0.000122    0.000092    0.000150        1.224490
23   currencyservice-1   trace.from_checkoutservice-2.duration_p95                          2       2022-03-21 00:33:00+08:00             0.000150    0.000107    0.000123    0.000092    0.000150        1.218522
53  currencyservice2-0        trace.from_frontend2-0.duration_mean                          2       2022-03-21 00:33:00+08:00             0.002917    0.001755    0.002513    0.001203    0.002917        1.160833
45   currencyservice-2         trace.from_frontend-1.duration_mean                          2       2022-03-21 00:49:00+08:00             0.000121    0.000094    0.000105    0.000085    0.000121        1.157341
54  currencyservice2-0         trace.from_frontend2-0.duration_p95                          2       2022-03-21 00:33:00+08:00             0.010802    0.007931    0.009363    0.007257    0.010802        1.153696
13   currencyservice-0         trace.from_frontend-2.duration_mean                          2       2022-03-21 00:39:00+08:00             0.000125    0.000096    0.000112    0.000089    0.000125        1.119570
42   currencyservice-2         trace.from_frontend-0.duration_mean                          1       2022-03-21 00:47:00+08:00             0.000126    0.000097    0.000113    0.000085    0.000126        1.115044
19   currencyservice-1  trace.from_checkoutservice-1.duration_mean                          2       2022-03-21 00:31:00+08:00             0.000130    0.000104    0.000117    0.000076    0.000130        1.112062
10   currencyservice-0         trace.from_frontend-1.duration_mean                          2       2022-03-21 00:30:00+08:00             0.000144    0.000098    0.000130    0.000084    0.000144        1.104294
33   currencyservice-1             trace.from_frontend-2.row_count                          2       2022-03-21 00:51:00+08:00            36.000000   24.000000   32.650000    7.000000   36.000000        1.102603
35   currencyservice-2   trace.from_checkoutservice-0.duration_p95                          1       2022-03-21 00:38:00+08:00             0.000124    0.000102    0.000113    0.000074    0.000124        1.097345
1    currencyservice-0   trace.from_checkoutservice-0.duration_p95                          2       2022-03-21 00:35:00+08:00             0.000126    0.000104    0.000116    0.000075    0.000126        1.086675
26   currencyservice-1          trace.from_frontend-0.duration_p95                          2       2022-03-21 00:33:00+08:00             0.000145    0.000115    0.000134    0.000103    0.000145        1.084112
12   currencyservice-0             trace.from_frontend-1.row_count                          2       2022-03-21 00:37:00+08:00            41.000000   27.500000   38.100000   12.000000   41.000000        1.076115
30   currencyservice-1             trace.from_frontend-1.row_count                          2       2022-03-21 00:37:00+08:00            41.000000   26.500000   38.100000   12.000000   41.000000        1.076115
47   currencyservice-2             trace.from_frontend-1.row_count                          2       2022-03-21 00:37:00+08:00            41.000000   26.500000   38.200000    3.000000   41.000000        1.073298```
```